# 01 - Data Exploration

## Objective
- Load `clinical_data.txt`
- Load `case_*.csv` files
- Inspect data structure
- Check missing values
- Check temporal length
- Create an initial simple plot

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
working_directory = Path.cwd().resolve()
candidates = [working_directory]
if working_directory.name == "notebooks":
    candidates.append(working_directory.parent)
required_files = (
    "pixi.toml", "src/data_utils.py",
    "notebooks/01_data_exploration.ipynb", "notebooks/02_metrics.ipynb",
)
project_root = next(
    (candidate for candidate in candidates
     if all((candidate / name).is_file() for name in required_files)),
    None,
)
if project_root is None:
    raise RuntimeError("Start this notebook from the project root or its notebooks/ directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(project_root)

In [ ]:
from src.data_utils import load_dataset

In [ ]:
# Reload module to avoid cache and ensure the edited version is used
from importlib import reload
import src.data_utils
reload(src.data_utils)
from src.data_utils import load_dataset
clinical, patients = load_dataset(data_path=project_root / "data/raw")
print("Clinical data shape:", clinical.shape)
print("Number of patients loaded:", len(patients))

In [ ]:
clinical.head()

In [ ]:
clinical.info()

In [ ]:
clinical.columns.tolist()

In [ ]:
patients[1].head()

In [ ]:
patients[1].info()

In [ ]:
patients[1].columns.tolist()

In [ ]:
print("Clinical columns:", clinical.columns.tolist())
print("Case 1 columns:", patients[1].columns.tolist())

In [ ]:
n_points = {case_id: len(df) for case_id, df in patients.items()}
n_points_series = pd.Series(n_points, name="n_points")

n_points_series.describe()

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(n_points_series, bins=20, edgecolor="black")
plt.xlabel("Number of CGM points")
plt.ylabel("Frequency")
plt.title("Distribution of number of points per patient")
plt.tight_layout()
plt.show()

In [ ]:
# Normalize patient case structure
import importlib
import src.data_utils as data_utils

importlib.reload(data_utils)
from src.data_utils import standardize_case

patients_std = {k: standardize_case(v) for k, v in patients.items()}

print(patients_std[1].head())
print(patients_std[1].columns.tolist())
print(patients_std[1].shape)

In [ ]:
# Are there duplicates in time?
patients_std[1]["time"].value_counts().head()

In [ ]:
# Are the timestamps duplicated? Or is it a bug in my code?
patients[1]["hora"].value_counts().head()

In [ ]:
print(len(patients[1]))
print(patients[1]["hora"].nunique())

In [ ]:
patients[1].groupby("hora")["glucemia"].nunique().value_counts()

### Key findings on CGM data quality

- Many timestamps are duplicated, but not all (331 unique times, 576 rows).
- Some time points have 2 values (duplicates), some 1 (normal), and some 0 (missing).
- The dataset has both duplicates and missing values—it's not fully clean.
- Next step: average duplicates and handle missing values before further analysis.

### Next steps: Data cleaning pipeline

- **Step 1:** Group duplicate timestamps and average glucose values.
- **Step 2:** Convert time column to datetime format.
- **Step 3:** Create a complete timeline with 5-minute intervals (ensures regular sampling, as required by the paper).
- **Step 4:** Measure missing values after reindexing.
- **Step 5:** Interpolate missing glucose values.

We will update the `standardize_case` function to implement this pipeline for each patient.

In [ ]:
patients_std[1].head()

In [ ]:
patients_std[1].head(10)

In [ ]:
patients_std[1].isna().sum()

In [ ]:
patients_std[1][patients_std[1]["glucose"].isna()].head(20)

In [ ]:
missing_summary = []

for case_id, df in patients_std.items():
    missing_glucose = df["glucose"].isna().sum()
    missing_pct = 100 * missing_glucose / len(df)

    missing_summary.append({
        "case_id": case_id,
        "n_points": len(df),
        "missing_glucose": missing_glucose,
        "missing_pct": missing_pct
    })

missing_df = pd.DataFrame(missing_summary)
missing_df.head()

In [ ]:
missing_df["missing_pct"].describe()

### Data quality summary and next steps

- We now have a clear table of missing data percentage per patient (e.g., case 1 → 0%, case 3 → 9%, etc.).
- Globally, the mean missingness is ~17.6%, median ~8.3%, but some patients have up to 99.6% missing data.
- This shows a very heterogeneous dataset: many patients are almost complete, some have moderate missingness, and a few are almost empty.
- This is typical for real CGM data and matches what the paper expects: a per-patient report for possible exclusion.
- **Next:** Decide whom to exclude based on missingness (e.g., <10%: good, 10–30%: borderline, >30%: exclude).

In [ ]:
# Classify patient data quality based on missing percentage

def classify_quality(pct):
    if pct < 10:
        return "good"
    elif pct < 30:
        return "medium"
    else:
        return "bad"

missing_df["quality"] = missing_df["missing_pct"].apply(classify_quality)
missing_df.head()

In [ ]:
missing_df["quality"].value_counts()

In [ ]:
# Check for missing values at the start and end of each patient series

def missing_edges(df, window=10):
    start_missing = df["glucose"].iloc[:window].isna().sum()
    end_missing = df["glucose"].iloc[-window:].isna().sum()
    return start_missing, end_missing

edge_summary = []
for case_id, df in patients_std.items():
    start_miss, end_miss = missing_edges(df)
    edge_summary.append({
        "case_id": case_id,
        "missing_start": start_miss,
        "missing_end": end_miss
    })

edges_df = pd.DataFrame(edge_summary)

# Combine with previous quality table
quality_full = missing_df.merge(edges_df, on="case_id")
quality_full.head()

In [ ]:
quality_full["final_quality"] = "bad"

quality_full.loc[
    (quality_full["missing_pct"] < 10) &
    (quality_full["missing_start"] == 0) &
    (quality_full["missing_end"] == 0),
    "final_quality"
] = "good"

quality_full.loc[
    (quality_full["missing_pct"] >= 10) &
    (quality_full["missing_pct"] < 30) &
    (quality_full["missing_start"] == 0) &
    (quality_full["missing_end"] == 0),
    "final_quality"
] = "medium"

In [ ]:
quality_full["final_quality"].value_counts()

In [ ]:
quality_full["has_edge_missing"] = (
    (quality_full["missing_start"] > 0) |
    (quality_full["missing_end"] > 0)
)

quality_full["has_edge_missing"].value_counts()

In [ ]:
pd.crosstab(quality_full["quality"], quality_full["has_edge_missing"])

A significant proportion of patients presented missing values at the beginning or end of the recording, which strongly affected data usability for time-series analysis methods such as DFA

### Comparing edge-missing detection with different window sizes

- We will check for missing values at the start and end of each patient series using two window sizes: 10 (strict, 50 min) and 3 (less strict, 15 min).
- Functions will be defined to compute edge missing, merge with quality, and classify final quality.
- We will compare the impact of both window sizes on patient quality classification, see which cases change, and decide which set to use for further analysis.
- This helps select a reasonable filter for general analysis (window=3) and a stricter one for DFA (window=10).

In [ ]:
# 1. Redefine missing_edges with window parameter
def missing_edges(df, window=10):
    start_missing = df["glucose"].iloc[:window].isna().sum()
    end_missing = df["glucose"].iloc[-window:].isna().sum()
    return start_missing, end_missing

# 2. Function to build edges_df
def build_edges_df(patients_dict, window=10):
    edge_summary = []
    for case_id, df in patients_dict.items():
        start_miss, end_miss = missing_edges(df, window=window)
        edge_summary.append({
            "case_id": case_id,
            "missing_start": start_miss,
            "missing_end": end_miss
        })
    return pd.DataFrame(edge_summary)

# 3. Function to build final quality table
def build_quality_full(missing_df, edges_df):
    quality_full = missing_df.merge(edges_df, on="case_id", how="left")
    quality_full["has_edge_missing"] = (
        (quality_full["missing_start"] > 0) |
        (quality_full["missing_end"] > 0)
    )
    def classify_final_quality(row):
        if row["has_edge_missing"]:
            return "bad"
        elif row["missing_pct"] < 10:
            return "good"
        elif row["missing_pct"] < 30:
            return "medium"
        else:
            return "bad"
    quality_full["final_quality"] = quality_full.apply(classify_final_quality, axis=1)
    return quality_full

# 4. Recalculate for window=10
edges_df_w10 = build_edges_df(patients_std, window=10)
quality_full_w10 = build_quality_full(missing_df, edges_df_w10)
quality_full_w10["final_quality"].value_counts()

In [ ]:
# 5. Recalculate for window=3
edges_df_w3 = build_edges_df(patients_std, window=3)
quality_full_w3 = build_quality_full(missing_df, edges_df_w3)
quality_full_w3["final_quality"].value_counts()

In [ ]:
# 6. Compare both results together
comparison = pd.DataFrame({
    "window_10": quality_full_w10["final_quality"].value_counts(),
    "window_3": quality_full_w3["final_quality"].value_counts()
}).fillna(0).astype(int)
comparison

In [ ]:
# 7. See how many have edge missing in each case
edge_compare = pd.DataFrame({
    "window_10": quality_full_w10["has_edge_missing"].value_counts(),
    "window_3": quality_full_w3["has_edge_missing"].value_counts()
}).fillna(0).astype(int)
edge_compare

In [ ]:
# 8. See which patients change category
quality_compare = quality_full_w10[["case_id", "final_quality"]].rename(
    columns={"final_quality": "quality_w10"}
).merge(
    quality_full_w3[["case_id", "final_quality"]].rename(
        columns={"final_quality": "quality_w3"}
    ),
    on="case_id"
)
quality_compare.head()

In [ ]:
# 8b. Patients that changed category
changed_cases = quality_compare[quality_compare["quality_w10"] != quality_compare["quality_w3"]]
changed_cases.head(20)

# How many changed?
len(changed_cases)

In [ ]:
# 9. Crosstab comparison
pd.crosstab(quality_compare["quality_w10"], quality_compare["quality_w3"])

In [ ]:
# 11. Build filtered datasets for each scenario
good_cases_basic = quality_full_w3[quality_full_w3["final_quality"] == "good"]["case_id"]
good_cases_dfa = quality_full_w10[quality_full_w10["final_quality"] == "good"]["case_id"]

print("Good basic:", len(good_cases_basic))
print("Good DFA:", len(good_cases_dfa))

patients_clean_basic = {k: v for k, v in patients_std.items() if k in good_cases_basic.values}
patients_clean_dfa = {k: v for k, v in patients_std.items() if k in good_cases_dfa.values}

In [ ]:
quality_full_w3["missing_start"].value_counts().head(10)

In [ ]:
quality_full_w3["missing_end"].value_counts().head(10)

### Interpretation of edge-missing results and recommended adjustment

- **Start missing:** Almost no patients have missing data at the beginning; the start is not a problem.
- **End missing:** More than half of patients have missing data at the end, often exactly 3 points (15 minutes).
- **Key insight:** The main issue is that many records end a bit early, not that there are gaps in the middle.
- **Current rule (missing_end == 0) is too strict**—it penalizes small, reasonable gaps at the end.
- **Recommended adjustment:** Allow up to 3 missing points at the end (15 minutes) for a patient to still be considered "good" quality.
- We will update the classification and compare the impact.

In [ ]:
# New recommended classification: allow up to 3 missing at end
def classify_final_quality(row):
    # allow up to 3 missing at the end
    if row["missing_start"] > 0:
        return "bad"
    elif row["missing_end"] > 3:
        return "bad"
    elif row["missing_pct"] < 10:
        return "good"
    elif row["missing_pct"] < 30:
        return "medium"
    else:
        return "bad"

quality_full_w3["final_quality_v2"] = quality_full_w3.apply(classify_final_quality, axis=1)
quality_full_w3["final_quality_v2"].value_counts()

In [ ]:
# Compare impact of new rule
pd.crosstab(
    quality_full_w3["final_quality"],
    quality_full_w3["final_quality_v2"]
)

## Final Dataset Selection and Preprocessing

After analyzing data quality, including missing values and edge effects, we defined a final filtering criterion to select reliable CGM recordings.

In particular:

- Patients with **less than 10% missing data** were considered acceptable.
- A tolerance of up to **3 missing points at the end** of the recording (≈15 minutes) was allowed.
- Patients with missing values at the beginning or larger gaps at the end were excluded.

This decision is motivated by the fact that short terminal gaps are common in CGM data and do not significantly affect global metrics, while missing values at the beginning or large gaps can distort time-series analyses.

Using this criterion, we selected a subset of high-quality patients for further analysis.

Next, we:

1. Filter the dataset to keep only valid patients.
2. Apply interpolation to handle remaining missing values.

In [ ]:
# Select good patients
good_cases = quality_full_w3[
    quality_full_w3["final_quality_v2"] == "good"
]["case_id"]

patients_clean = {
    k: v for k, v in patients_std.items() if k in good_cases.values
}

print("Number of selected patients:", len(patients_clean))

In [ ]:
# Function to interpolate glucose values in a DataFrame
def interpolate_glucose(df):
    df = df.copy()
    df["glucose"] = df["glucose"].interpolate()
    return df

In [ ]:
# Interpolate glucose for all patients in patients_clean
patients_interp = {
    k: interpolate_glucose(v)
    for k, v in patients_clean.items()
}

In [ ]:
# Check missing after interpolation
for k in list(patients_interp.keys())[:3]:
    print(f"Patient {k} missing:", patients_interp[k]["glucose"].isna().sum())

In [ ]:
total_missing = sum(df["glucose"].isna().sum() for df in patients_interp.values())
print("Total missing after interpolation:", total_missing)

In [ ]:
#  Save each interpolated patient to CSV
import os
output_path = project_root / "data/processed/interpolated"
os.makedirs(output_path, exist_ok=True)
for k, df in patients_interp.items():
    df.to_csv(f"{output_path}/case_{k}.csv", index=False)

#  Save quality summary table
quality_full_w3.to_csv(project_root / "data/processed/quality_summary.csv", index=False)